# Content Understanding via governed APIM gateway

## How the CU endpoint works

Azure AI Content Understanding is a capability of an **Azure AI Services account**
(`kind: AIServices`), not a separate resource. In this example it is hosted on
`aif-cu-{suffix}` (deployed into `rg-foundry-cu-{suffix}`), accessible at:

```
https://aif-cu-{suffix}.cognitiveservices.azure.com/contentunderstanding/...
```

That same account hosts local deployments of `gpt-4.1-mini` and
`text-embedding-3-large`, which CU field extraction analyzers call internally.
The `cu-project` Foundry project exists only to hold the APIM connection - it
plays no direct role in CU analysis.

All calls in this notebook go through the core APIM `/cu` API, which proxies to
the CU endpoint using managed identity:

```
This notebook  →  APIM /cu  →  aif-cu-{suffix}/contentunderstanding/...
               api-key auth     managed-identity auth
```

## Demonstrations

1. **List analyzers** - enumerate all prebuilt analyzers available on the account
2. **Analyze a document** - submit a PDF for async analysis via `prebuilt-layout`
3. **Poll results** - rewrite the `Operation-Location` URL to route through APIM and poll

All HTTP calls use `CU_GATEWAY_KEY` - no `DefaultAzureCredential` is needed.

## Prerequisites

- The APIM resource must have been successfully deployed in the earlier project setup.
- `.env` must contain `CU_GATEWAY_KEY`, `CU_FOUNDRY_PROJECT_ENDPOINT`, `CU_APIM_CONNECTION`

## Setup: imports and env load

In [1]:
import os
import json
import subprocess
import time
import urllib.request
from pathlib import Path
from IPython.display import clear_output, Markdown

repo_root = Path(
    subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip()
)
env_file = repo_root / '.env'

with open(env_file) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ[key] = value

GATEWAY_URL                 = os.environ['GATEWAY_URL']
CU_GATEWAY_KEY              = os.environ['CU_GATEWAY_KEY']
CU_FOUNDRY_PROJECT_ENDPOINT = os.environ['CU_FOUNDRY_PROJECT_ENDPOINT']
CU_APIM_CONNECTION          = os.environ['CU_APIM_CONNECTION']
CHAT_MODEL                  = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')

# Derive the CU APIM base URL from the main gateway URL
# https://apim-foundry-{suffix}.azure-api.net/openai  ->  .../cu
CU_GATEWAY_URL = GATEWAY_URL.rstrip('/').rsplit('/openai', 1)[0] + '/cu'
CU_API_VERSION = '2025-11-01'

print(f'CU gateway URL   : {CU_GATEWAY_URL}')
print(f'CU API version   : {CU_API_VERSION}')
print(f'APIM connection  : {CU_APIM_CONNECTION}')
print(f'Gateway key      : {CU_GATEWAY_KEY[:4]}... (hidden)')

CU gateway URL   : https://apim-foundry-6fe574.azure-api.net/cu
CU API version   : 2025-11-01
APIM connection  : landing-zone-apim
Gateway key      : 016a... (hidden)


## Step 1: List available analyzers

In [2]:
list_url = f'{CU_GATEWAY_URL}/analyzers?api-version={CU_API_VERSION}'

req = urllib.request.Request(
    list_url,
    headers={'api-key': CU_GATEWAY_KEY}
)

with urllib.request.urlopen(req) as resp:
    assert resp.status == 200, f'Expected 200, got {resp.status}'
    analyzers_data = json.loads(resp.read())

analyzers = analyzers_data.get('value', [])
print(f'Total analyzers: {len(analyzers)}')
print()
print('First 10 analyzers:')
for a in analyzers[:10]:
    print(f'  {a["analyzerId"]:40s}  {a.get("description", "")[:60]}')

analyzer_ids = [a['analyzerId'] for a in analyzers]
assert 'prebuilt-layout' in analyzer_ids, 'prebuilt-layout not found in analyzer list'
print('\nprebuilt-layout: OK')

Total analyzers: 87

First 10 analyzers:
  prebuilt-audio                            Transcribe conversations.
  prebuilt-audioSearch                      Transcribe conversations and extract summaries.
  prebuilt-bankStatement.us                 Extract bank statement US document fields.
  prebuilt-callCenter                       Analyze call center conversations to extract transcripts, su
  prebuilt-check.us                         Extract check US document fields.
  prebuilt-contract                         Extract contract document fields.
  prebuilt-creditCard                       Extract credit card document fields.
  prebuilt-creditMemo                       Extract credit memo document fields.
  prebuilt-document                         Extract various content and layout elements such as words, p
  prebuilt-documentFields                   Propose key-value document fields.

prebuilt-layout: OK


## Step 2: Analyze a document

Submit a PDF for async analysis using `prebuilt-layout`. The request body is
`{"url": "..."}` - a flat structure, not nested in an `inputs` array.

The CU service returns 202 Accepted with an `Operation-Location` header pointing to
the result polling URL - we rewrite this URL to route through APIM.

In [6]:
import urllib.error

ANALYZER_ID  = 'prebuilt-layout'
# Attention Is All You Need - reliably accessible public PDF
DOCUMENT_URL = 'https://arxiv.org/pdf/1706.03762'

analyze_url  = f'{CU_GATEWAY_URL}/analyzers/{ANALYZER_ID}:analyze?api-version={CU_API_VERSION}'
analyze_body = json.dumps({
    'inputs': [{'url': DOCUMENT_URL}]
}).encode('utf-8')

req = urllib.request.Request(
    analyze_url,
    data=analyze_body,
    method='POST',
    headers={
        'api-key':      CU_GATEWAY_KEY,
        'Content-Type': 'application/json',
    }
)

try:
    with urllib.request.urlopen(req) as resp:
        assert resp.status == 202, f'Expected 202, got {resp.status}'
        operation_location = resp.headers.get('Operation-Location', '')
except urllib.error.HTTPError as e:
    print(f'HTTP {e.code}: {e.read().decode()}')
    raise

print(f'Analysis submitted (202 Accepted)')
print(f'Operation-Location: {operation_location[:80]}...')

# Rewrite the polling URL to route through APIM instead of hitting CU directly.
# The Operation-Location header points to cognitiveservices.azure.com - extract
# the path segment after /contentunderstanding and prepend the APIM CU gateway URL.
if 'cognitiveservices.azure.com' in operation_location:
    path_after_cu = operation_location.split('/contentunderstanding', 1)[1]
    poll_url = CU_GATEWAY_URL + path_after_cu
else:
    # Already an APIM URL (e.g. on re-run) - use as-is
    poll_url = operation_location

print(f'\nRewritten poll URL : {poll_url[:80]}...')

Analysis submitted (202 Accepted)
Operation-Location: https://aif-cu-ii5drx.cognitiveservices.azure.com/contentunderstanding/analyzerR...

Rewritten poll URL : https://apim-foundry-6fe574.azure-api.net/cu/analyzerResults/4cd6d816-a83d-43e5-...


## Step 3: Poll for results

In [7]:
MAX_WAIT_SECONDS = 300
POLL_INTERVAL    = 3
elapsed          = 0
result           = None

while elapsed < MAX_WAIT_SECONDS:
    req = urllib.request.Request(
        poll_url,
        headers={'api-key': CU_GATEWAY_KEY}
    )
    with urllib.request.urlopen(req) as resp:
        poll_data = json.loads(resp.read())

    status = poll_data.get('status', 'Unknown')
    clear_output(wait=True)
    print(f'Status: {status}  (elapsed: {elapsed}s)')

    if status in ('Succeeded', 'Failed', 'Cancelled'):
        result = poll_data
        break

    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL

if result is None:
    raise RuntimeError(f'Analysis did not complete within {MAX_WAIT_SECONDS}s')
if result.get('status') != 'Succeeded':
    raise RuntimeError(f'Analysis failed with status: {result.get("status")}\n{json.dumps(result, indent=2)}')

print(f'Analysis complete: {result["status"]}')

Status: Succeeded  (elapsed: 0s)
Analysis complete: Succeeded


## Results

In [8]:
assert result['status'] == 'Succeeded', f'Expected Succeeded, got {result["status"]}'
assert len(result['result']['contents']) > 0, 'No contents in result'

markdown_content = result['result']['contents'][0]['markdown']
print(f'Extracted markdown length: {len(markdown_content):,} characters')
print(f'Content items: {len(result["result"]["contents"])}')
print()

Markdown(markdown_content[:3000])

Extracted markdown length: 48,935 characters
Content items: 1



<!-- PageHeader: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. -->


# Attention Is All You Need

Ashish Vaswani*
Google Brain
avaswani@google.com

Noam Shazeer*
Google Brain
noam@google.com

Niki Parmar*
Google Research
nikip@google.com

Jakob Uszkoreit*
Google Research
usz@google.com

Llion Jones*
Google Research
llion@google.com

Aidan N. Gomez* +
University of Toronto
aidan@cs.toronto.edu

Łukasz Kaiser*
Google Brain
lukaszkaiser@google.com

Illia Polosukhin* $
illia.polosukhin@gmail.com


## Abstract

The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training data.

<!-- PageFooter: *Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started the effort to evaluate this idea. Ashish, with Illia, designed and implemented the first Transformer models and has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head attention and the parameter-free position representation and became the other person involved in nearly every detail. Niki designed, implemented, tuned and evaluated countless model variants in our original codebase and tensor2tensor. Llion also experimented with novel model variants, was responsible for our initial codebase, and efficient inference and visualizations. Lukasz and Aidan spent countless long days designing various parts of and implementing tensor2tensor, replacing our earlier codebase, greatly improving results and massively accelerating our research. -->
<!-- PageFooter: +Work performed while at Google Brain. -->
<!-- PageFooter: #Work performed while at Google Research. -->
<!-- PageFooter: 31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA. -->

arXiv:1706.03762v7 [cs.CL] 2 Aug 2023

<!-- PageBreak -